# Import de PySpark et des clés

In [1]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import numpy as np
from datetime import datetime, date

import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql import Row
spark = SparkSession.builder \
    .appName("BGES") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

In [2]:
import pandas as pd 
mat_inf = pd.read_csv('BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BERLIN_INFORMATIQUE/MATERIEL_INFORMATIQUE_20260429.txt',sep=';').columns
personnel = pd.read_csv('BDD_BGES/BDD_BGES_BERLIN/PERSONNEL_BERLIN.txt',sep=';').columns
mission = pd.read_csv('BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BERLIN_MISSION/MISSION_20260429.txt',sep=';').columns
print(pd.read_csv('BDD_BGES/BDD_BGES_BERLIN/PERSONNEL_BERLIN.txt',sep=';')['TS_CREATION_PERSONNEL'])
mat_inf = list(mat_inf)
personnel = list(personnel)
mission = list(mission)

KeyPersoMission = []
for i in personnel :
    if i in mission : 
        KeyPersoMission.append(i)

KeyPersoMission

0       2000-05-09 11:54:25
1       2014-08-17 01:27:04
2       2013-01-22 14:11:50
3       2007-08-27 01:28:18
4       2012-03-02 18:24:08
               ...         
4238    2002-07-10 13:02:23
4239    2023-06-25 19:12:37
4240    2020-06-27 04:11:52
4241    2007-06-06 12:15:27
4242    2003-10-02 11:51:43
Name: TS_CREATION_PERSONNEL, Length: 4243, dtype: object


['ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL']

In [3]:
print(mat_inf)
print(personnel)
print(mission)

['ID_MATERIELINFO', 'ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL', 'DATE_ACHAT', 'TYPE', 'MODELE']
['ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL', 'DT_NAISS', 'VILLE_NAISS', 'PAYS_NAISS', 'NUM_SECU', 'IND_PAYS_NUM_TELP', 'NUM_TELEPHONE', 'NUM_VOIE', 'DSC_VOIE', 'CMPL_VOIE', 'CD_POSTAL', 'VILLE', 'PAYS', 'FONCTION_PERSONNEL', 'TS_CREATION_PERSONNEL', 'TS_MAJ_PPERSONNEL']
['ID_MISSION', 'ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL', 'DATE_MISSION', 'TYPE_MISSION', 'VILLE_DEPART', 'PAYS_DEPART', 'VILLE_DESTINATION', 'PAYS_DESTINATION', 'TRANSPORT', 'ALLER_RETOUR']


# Tables
- Faits
    - ID_MISSION
    - ID_PERSONNEL
    - ID_MATERIELINFO
- Trajet : ID_MISSION
    - MISSION : ID_MISSION, PAYS_DEPART, PAYS_DESTINATION, VILLE_DEPART, VILLE_DESTINATION,
- Personnel - ID_PERSONNEL
    - ID_PERSONNEL, VILLE, PAYS, FONCTION, TS_CREATION_PERSONNEL, TS_MAJ_PPERSONNEL
- MatInf - ID_MATERIELINFO
    - Matériel informatique : ID_MATERIELINFO, DATE_ACHAT, TYPE, MODELE
    - Materiel informatique impact : Impact (join on type and modèle)

In [7]:
import glob
#Charger les fichiers de personnel
fichiers_personnel = glob.glob('BDD_BGES/**/PERSONNEL_*.txt', recursive=True)
psdf_personnel_list = [ps.read_csv(f, sep=';',index_col='ID_PERSONNEL') for f in fichiers_personnel]
psdf_personnel = ps.concat(psdf_personnel_list, ignore_index=False)

#Charger les fichiers de matériel informatique
fichiers_mat_inf = glob.glob('BDD_BGES/**/MATERIEL_INFORMATIQUE_*.txt', recursive=True)
psdf_mat_inf_list = [ps.read_csv(f, sep=';') for f in fichiers_mat_inf]
psdf_mat_inf_temp = ps.concat(psdf_mat_inf_list, ignore_index=True)

#Charge les données d'impact carbone du matériel informatique
psdf_impact_matinf = ps.read_csv('BDD_BGES/materiel_informatique_impact.csv',sep=',')
psdf_impact_matinf = psdf_impact_matinf.rename(columns={
    'Type': 'TYPE',
    'Impact': 'IMPACT',
    'Modèle': 'MODELE'
})

#Fusionne les données de matériel informatique avec les données d'impact carbone
psdf_mat_inf = psdf_mat_inf_temp.merge(
    psdf_impact_matinf,
    on=['TYPE','MODELE'],
    how='inner'
)
psdf_mat_inf = psdf_mat_inf.set_index('ID_MATERIELINFO')

#Charger les fichiers de mission
fichiers_mission = glob.glob('BDD_BGES/**/MISSION_*.txt', recursive=True)
psdf_mission_list = [ps.read_csv(f, sep=';',index_col='ID_MISSION') for f in fichiers_mission]
psdf_mission = ps.concat(psdf_mission_list, ignore_index=False)

c:\Users\rzong\anaconda3\envs\nf26_env\Lib\site-packages\pyspark\pandas\utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [11]:
# Transformation en spark dataframe
from pyspark.sql.functions import *
sdf_personnel = psdf_personnel.to_spark(index_col='ID_PERSONNEL')
sdf_personnel.show(10)
sdf_mat_inf = psdf_mat_inf.to_spark(index_col='ID_MATERIELINFO')
sdf_mat_inf.show(10)
sdf_mission = psdf_mission.to_spark(index_col='ID_MISSION')
sdf_mission.show(10)

+--------------------+-------------+----------------+----------+------------+----------+-----------+-----------------+-------------+--------+----------+---------+---------+------+-------+------------------+---------------------+-------------------+
|        ID_PERSONNEL|NOM_PERSONNEL|PRENOM_PERSONNEL|  DT_NAISS| VILLE_NAISS|PAYS_NAISS|   NUM_SECU|IND_PAYS_NUM_TELP|NUM_TELEPHONE|NUM_VOIE|  DSC_VOIE|CMPL_VOIE|CD_POSTAL| VILLE|   PAYS|FONCTION_PERSONNEL|TS_CREATION_PERSONNEL|  TS_MAJ_PPERSONNEL|
+--------------------+-------------+----------------+----------+------------+----------+-----------+-----------------+-------------+--------+----------+---------+---------+------+-------+------------------+---------------------+-------------------+
|KeyPers_Berlin_12...|        Name0|       FistName0|1948-11-17|      Bogota|  Colombia|NS000000000|             NULL|+336##0530601|      51|NomVoie520|     NULL|    #8356|Berlin|Germany|    Dateningenieur|  2000-05-09 11:54:25|2000-05-09 11:54:25|
|Key

In [14]:
#Création du modèle étoile
Faits = sdf_mission.join(sdf_personnel, on='ID_PERSONNEL', how='inner').join(sdf_mat_inf, on='ID_PERSONNEL', how='inner').select('ID_MISSION','ID_PERSONNEL','ID_MATERIELINFO')
Faits.show(10)
Trajets = sdf_mission.select('ID_MISSION', 'PAYS_DEPART', 'PAYS_DESTINATION', 'VILLE_DEPART', 'VILLE_DESTINATION')
Trajets.show(10)
Personnel = sdf_personnel.select('ID_PERSONNEL', 'VILLE', 'PAYS', 'FONCTION_PERSONNEL', 'TS_CREATION_PERSONNEL', 'TS_MAJ_PPERSONNEL')
Personnel.show(10)
MaterielInformatique = sdf_mat_inf.select('ID_MATERIELINFO', 'DATE_ACHAT', 'TYPE', 'MODELE', 'IMPACT')
MaterielInformatique.show(10)

+-----------------+--------------------+--------------------+
|       ID_MISSION|        ID_PERSONNEL|     ID_MATERIELINFO|
+-----------------+--------------------+--------------------+
|BERLIN_2026043020|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...|
|BERLIN_2026043017|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...|
|BERLIN_2026050123|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...|
| BERLIN_202605011|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...|
|BERLIN_2026050120|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...|
| BERLIN_202605026|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...|
|BERLIN_2026050218|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...|
|BERLIN_2026050411|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...|
| BERLIN_202605047|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...|
|BERLIN_2026050513|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...|
+-----------------+--------------------+--------------------+
only showing top 10 rows
+----------------+-----------+----------------+------------+-----------------+
|      ID_MISSION|PAYS_DEPAR

In [ ]:
from pyspark.sql.functions import col, count, when, current_date, trim

# Fonction pour compter les valeurs nulles
def count_nulls(df, name):
    print(f"Null values dans {name}:")
    df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show(truncate=False)
    print("\n")

# Helper pour détecter les chaînes vides ou nulles
def is_blank(column):
    return column.isNull() | (trim(column) == "")

# Fonction pour afficher les lignes suspectes selon un filtre
def show_suspect_rows(df, name, condition, message):
    print(f"{name} - {message}")
    df.filter(condition).show(20, truncate=False)
    print("\n")

# Comptage global des valeurs manquantes
count_nulls(Faits, "Faits")
count_nulls(Trajets, "Trajets")
count_nulls(Personnel, "Personnel")
count_nulls(MaterielInformatique, "MaterielInformatique")

# Recherche des valeurs aberrantes ou incohérences dans les tables dérivées
show_suspect_rows(
    Faits,
    "Faits",
    col("ID_MISSION").isNull() | col("ID_PERSONNEL").isNull() | col("ID_MATERIELINFO").isNull(),
    "lignes avec clés manquantes"
)

show_suspect_rows(
    Trajets,
    "Trajets",
    is_blank(col("PAYS_DEPART")) |
    is_blank(col("PAYS_DESTINATION")) |
    is_blank(col("VILLE_DEPART")) |
    is_blank(col("VILLE_DESTINATION")),
    "trajets avec pays ou villes manquants"
)

show_suspect_rows(
    Trajets,
    "Trajets",
    col("VILLE_DEPART") == col("VILLE_DESTINATION"),
    "trajets où la ville de départ est identique à la ville de destination"
)

show_suspect_rows(
    Personnel,
    "Personnel",
    is_blank(col("VILLE")) |
    is_blank(col("PAYS")) |
    is_blank(col("FONCTION_PERSONNEL")) |
    (col("TS_CREATION_PERSONNEL") > current_date()) |
    (col("TS_MAJ_PPERSONNEL") > current_date()),
    "personnel avec informations manquantes ou dates futures"
)

show_suspect_rows(
    MaterielInformatique,
    "MaterielInformatique",
    is_blank(col("TYPE")) |
    is_blank(col("MODELE")) |
    col("IMPACT").isNull() |
    (col("IMPACT") <= 0) |
    (col("DATE_ACHAT") > current_date()),
    "matériel avec type/modèle manquant, impact incorrect ou date d'achat future"
)

Missing values dans Faits:
+----------+------------+---------------+
|ID_MISSION|ID_PERSONNEL|ID_MATERIELINFO|
+----------+------------+---------------+
|0         |0           |0              |
+----------+------------+---------------+



Missing values dans Trajets:
+----------+-----------+----------------+------------+-----------------+
|ID_MISSION|PAYS_DEPART|PAYS_DESTINATION|VILLE_DEPART|VILLE_DESTINATION|
+----------+-----------+----------------+------------+-----------------+
|0         |0          |0               |0           |0                |
+----------+-----------+----------------+------------+-----------------+



Missing values dans Personnel:
+------------+-----+----+------------------+---------------------+-----------------+
|ID_PERSONNEL|VILLE|PAYS|FONCTION_PERSONNEL|TS_CREATION_PERSONNEL|TS_MAJ_PPERSONNEL|
+------------+-----+----+------------------+---------------------+-----------------+
|0           |0    |0   |0                 |0                    |0          

PySparkValueError: [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.

## ETL : Formatisation des données
#### A faire : 
- Tout traduire en anglais (exemple la fonction de directeur à berlin s'appelle Führungskraft)

## Requête sur le DataWarehouse

#### 1. Combien de cadres travaillent sur le site de Paris?

#### 2. Combien d’ingénieurs Data travaillent sur les sites aux États-Unis ?

#### 3. Combien d’ingénieurs informaticiens travaillent dans l’organisation (tous sites compris) ?

#### 4. Combien de PC fixes ont été achetés par l’organisation entre juin et septembre 2026 ?

#### 5. Quelle a été l’impact carbone des PC fixes sans ecran entre mai et octobre 2026 ?

#### 6. Quel a été l’impact carbone des PC portables achetés par les ingénieurs Data entre mai et octobre 2026 sur les sites de Londres et New-York?

#### 7. Quel a été l’impact carbone des écrans achetés par les cadres entre juillet et septembre 2026 sur tous les sites de l’organisation ?

#### 8. Quel a été l’impact carbone des missions sur les sites Européens entre mai et octobre 2026 ?

#### 9. Quels ont été les 5 jours les plus impactants concernant les missions en avion pour les sites Européens de l’organisation ?

#### 10. Quel a été le secteur d’activité qui a eu le plus d’impact concernant les missions et le matériel informatique sur l’ensemble des sites de l’organisation ?

#### 11.  Quel site a eu le plus d’impact concernant les missions et le matériel informatique sur l’ensemble des sites de l’organisation ?

#### 12. Quel a été l’impact carbone des missions reliant chaque site (la ville de départ est un site de l’organisation et la ville d’arrivée également) durant le mois de septembre 2026 ?

#### 13. Quel a été l’impact carbone des séminaires en juillet 2026 pour les employés de Los Angeles ?

#### 14. Quel secteur d’activité a été le plus impactant pour les missions “conférences” entre mai et septembre 2026 ?

#### 15. Quel a été l’âge moyen des employés Ingénieurs Data qui sont partis en formations entre juillet et septembre 2026 ?

#### 16. Quelle destination a été la plus impactante (en cumul) entre mai et octobre 2026 ?

#### 17. Quelles ont été les trois catégories de missions les plus impactantes pour les cadres dans les sites Européens en mai 2026 ?

*Les réponses des questions suivantes devront être sous forme de figures*
#### 18. Quelles ont été les 5 missions les plus impactantes sur le site de Paris ?

#### 19. Proposer une figure comparant l’impact carbone mensuel des missions en fonction du type de transport et sur chaque site.

#### 20. Proposer une figure illustrant l’impact carbone global mensuel de l’organisation.